# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [32]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
import os, sys
from langchain_community.document_loaders import PyPDFLoader

# print (PyPDFLoader.__doc__) # check exists.

# made docs folder
from pathlib import Path
path = os.getcwd()
folder = 'doc'
# print (path + '/' +folder)
new_directory_path = Path(path + '/' + folder)

if not os.path.exists (folder):
    new_directory_path.mkdir(parents=True, exist_ok=True)
    print (f"folder made")

# get doc from online 

## Select a Document


from urllib.request import urlretrieve

url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
filename = "ai_report_2025.pdf"
path = path + '/' + folder
fullpath = path + '/' + filename
print (fullpath)

def download_pdf_urllib(url, fullpath):

    try:
        urlretrieve(url, fullpath)
        print(f"Successfully downloaded '{filename}' using urllib")
    except Exception as e:
        print(f"An error occurred: {e}")

download_pdf_urllib(url, fullpath)


# now get the doc length
from langchain_community.document_loaders import PyPDFLoader

# file_path = "../example_data/nke-10k-2023.pdf"
loader = PyPDFLoader(fullpath)

docs = loader.load()

print(len(docs))

# now get content
print(docs[0].metadata)

if docs:
    metadata = docs[0].metadata
    #print(first_page_metadata)
    # The author is usually under 'author' or 'creators'
    author = metadata.get('author')
    title = metadata.get('title')
    print(f"Author: {author}")
    print(f"Title: {title}")



folder made
/Users/darko-adm/work/dsi/deploying-ai/02_activities/doc/ai_report_2025.pdf
Successfully downloaded 'ai_report_2025.pdf' using urllib
26
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': '/Users/darko-adm/work/dsi/deploying-ai/02_activities/doc/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}
Author: Aditya Challapally
Title: None


In [7]:

# now get the doc length
from langchain_community.document_loaders import PyPDFLoader

# file_path = "../example_data/nke-10k-2023.pdf"
loader = PyPDFLoader(fullpath)

docs = loader.load()

print(len(docs))

# now get content
print(docs[0].metadata)

if docs:
    metadata = docs[0].metadata
    #print(first_page_metadata)
    # The author is usually under 'author' or 'creators'
    author = metadata.get('author')
    title = metadata.get('title')
    print(f"Author: {author}")
    print(f"Title: {title}")

    # Grab the first page
first_page_text = docs[0].page_content

# The title is usually the first non-empty line
title_fallback = first_page_text.strip().split('\n')[3]
print(f"Likely Title: {title_fallback}")


#print(list(docs[0].metadata))

26
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': '/Users/darko-adm/work/dsi/deploying-ai/02_activities/doc/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}
Author: Aditya Challapally
Title: None
Likely Title: The GenAI Divide  


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [34]:
from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

models = client.models.list()

for model in models:
   print (model)


# add the text of my pdf doc. chunk it into smaller sections.
paragraphs = [p.strip() for p in full_text.split("\n\n") if p.strip()]

#messages = [{"role": "user", "content": p} for p in paragraphs]
#print (messages)
# chunked_text = "\n\n".join(paragraphs)
#print (chunked_text)
#raw_text = "\n\n".join(d.page_content for d in docs)
#paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
#full_doc = "\n\n".join(paragraphs)
full_doc = "\n\n".join(d.page_content for d in docs)


# get summary of document.
#for i, para in enumerate(paragraphs):
#    response = client.responses.create(
##        model="gpt-4o",
#        input=[{"role": "user", "content": summary_prompt1 + "\n\n" + full_doc}]
#    )

total_input_tokens = 0
total_output_tokens = 0

title_page = first_page_text.strip().split('\n')[3]


author_prompt = "You are a professional guidance counselor at a university with over 10 years experience at the job. " \
    "Analyze the included document and can you tell me know the author or authors are of this document? "

response = client.responses.create(
    model="gpt-4o-mini",
    input=[
          {"role": "system", "content": author}, 
        {"role": "user", "content": title_page}], max_output_tokens=500
)

author = response.output_text
usage = response.usage
print (usage.output_tokens)
total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens

title_prompt = "You are a professional guidance counselor at a university with over 10 years experience at the job. " \
    "Analyze the included document and can you tell what is the exact title of this document? "

response = client.responses.create(
    model="gpt-4o-mini",
    input=[
          {"role": "system", "content": title_prompt}, 
        {"role": "user", "content": title_page}], max_output_tokens=500
)

title = response.output_text
print (usage.output_tokens)
total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens




relevance_prompt = "You are a professional guidance counselor at a university with over 10 years experience at the job. " \
    "Analyze the included document and explain to me why this document is important and relevant " \
    "for an AI professional in their professional development. " \
    "Provide your summary in one paragraph length."

response = client.responses.create(
    model="gpt-4o",
    input=[
          {"role": "system", "content": relevance_prompt}, 
        {"role": "user", "content": full_doc}], max_output_tokens=500
)

usage = response.usage

relevance = response.output_text
print (usage.output_tokens)
total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens

user_content = full_doc
summary_prompt = "You are a professional guidance counselor at a university with over 10 years experience at the job. " \
    "Summarize the following document using the tonal format of African-American Vernacular English. "

response = client.responses.create(
    model="gpt-4o", 
    input=[
        {"role": "system", "content": summary_prompt}, 
        {"role": "user", "content": full_doc}], max_output_tokens=1000
)

usage = response.usage

summary = response.output_text
print (usage.output_tokens)
total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens


tone_prompt = "Explin to me in detail the tone used in crafting this prompt." 

response = client.responses.create(
    model="gpt-4o", 
    input=[
        {"role": "user", "content": tone_prompt},
        {"role": "user", "content": full_doc}], max_output_tokens=500
)

usage = response.usage

tone = response.output_text

total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens

from IPython.display import display, Markdown
display(Markdown(f"Author: {author}"))
display(Markdown(f"Title: {title}"))
display(Markdown(f"Relevance: {relevance}"))
display(Markdown(f"Summary: {summary}"))
display(Markdown(f"Tone: {tone}"))
display(Markdown(f"Total input tokens: {total_input_tokens}"))
display(Markdown(f"Total output tokens: {total_output_tokens}"))


#response = client.responses.create(
##    model = 'gpt-4o',
#    input = docs
#)

#from IPython.display import display, Markdown
#display(Markdown(response.output_text))

Model(id='text-embedding-3-small', created=1705948997, object='model', owned_by='system')
Model(id='gpt-4o', created=1715367049, object='model', owned_by='system')
Model(id='gpt-4o-mini', created=1721172741, object='model', owned_by='system')
Model(id='gpt-5', created=1754425777, object='model', owned_by='system')
429
429
163
355


Author: "The GenAI Divide" report highlights the current state of generative AI (GenAI) in business, presenting insights into the challenges and disparities in its adoption and effectiveness. Here’s a summary of the key points:

### Overview
Despite significant investments in GenAI—estimated at $30-40 billion—most organizations are not realizing substantial performance improvements, with 95% reporting no measurable impact.

### Key Findings

1. **Adoption vs. Transformation**:
   - Many companies use generative tools (like ChatGPT), but few see transformative results. The Technology and Media sectors lead in impactful changes, while others lag behind.

2. **Pilot Challenges**:
   - Numerous GenAI pilot projects fail to scale due to integration issues and fragile workflows, leading to many enterprise systems being rejected.

3. **Learning Gap**:
   - The primary obstacle isn't a lack of resources but the absence of adaptive systems. Many existing tools lack context retention and evolution capabilities.

4. **Investment Trends**:
   - More than half of GenAI spending goes to sales and marketing, which often yields lower returns compared to back-office automation investments.

5. **Successful Strategies**:
   - Firms that excel in integrating GenAI do so by deeply customizing tools and empowering line managers to work collaboratively with vendors.

6. **Shadow AI Economy**:
   - Employees often use personal AI tools outside official programs, often achieving better results and ROI.

7. **Workforce Impact**:
   - GenAI is unlikely to cause widespread layoffs but may shift hiring towards candidates with AI skills and lead to reduced outsourcing of lower-value tasks.

### Recommendations

- **Prioritize Custom Solutions**: Organizations should buy tailored tools rather than build their own, as partnerships tend to yield better outcomes.
  
- **Decentralize Decision-Making**: Frontline managers should be empowered to source and implement AI solutions.

- **Invest in Operations**: Focus on back-office applications that boost efficiency for better ROI compared to front-office solutions.

By implementing these strategies, organizations can navigate the GenAI landscape more effectively and begin to close the divide between adoption and meaningful impact.

Title: The title of the document is "The GenAI Divide." If you have specific questions or need assistance with its content, feel free to ask!

Relevance: This document, "The GenAI Divide," is crucial for AI professionals as it highlights the challenges and strategies related to AI deployment in businesses by 2025. It identifies a disparity—termed the GenAI Divide—where only 5% of AI implementations achieve significant business transformation. The report emphasizes that the core issue isn't technology but the lack of adaptive learning within systems. It suggests successful AI deployment relies on external partnerships, deep customization, and focusing on workflow integration rather than generic solutions. Additionally, the text explores the importance of employee-driven adoption, outlines potential organizational changes, and introduces the concept of an Agentic Web, hinting at future trends in AI integration. An AI professional would find this document valuable for understanding current industry challenges, strategic solutions, and the evolving landscape of AI technology in enterprise settings.

Summary: Alright, so here’s the lowdown on what that document’s breakin' down:

The GenAI Divide talkin' 'bout how even though there's a whole lotta money bein' thrown at AI, only a lil’ piece of it actually makin' big bucks. Most folks usin’ AI don’t see no real change, just small boosts in work like ChatGPT helpin’ with productivity but not really changin’ the game. Big companies start with a lotta interest, but only a few actually make it work in a big way.

They say the problem ain’t the tech or rules, but how folks approach it. GenAI tools ain’t learnin’ and growin’, so they just ain't hittin' the mark. Some folks got it right by realizin' that, demandin' systems that fit their biz and improve over time. This helps 'em save money and get better at holdin’ on to customers.

Now, there's this “shadow AI economy” where people are takin’ personal AI tools to work even though their bosses’ setups ain’t crossin' over from trials to everyday use. Folks trust what they know works.

Investments lean heavy on sales and marketing 'cause it’s easy to show numbers there, but the real cash might be in automatin’ behind-the-scenes work like finance.

To cross over to the good side of the GenAI Divide, folks gotta stop buildin’ their own stiff systems and start buyin’ tools that actually learn and get better. Winners will be those who adjust tech to real business needs, not just the flashiest tech out there.

So, the takeaway? Get on those learnin’, adaptable systems, and the right partnerships to really get value from AI.

Tone: The tone of the document is analytical and professional. It approaches the subject with an objective and methodical perspective, focusing on empirical data and research findings. The language is formal, precise, and intended for a knowledgeable audience. Key elements include:

1. **Objective Analysis**: The document presents findings from structured interviews, surveys, and systematic reviews, emphasizing data-driven insights.

2. **Neutral and Disclaimer-Heavy**: It includes disclaimers about the views expressed and emphasizes confidentiality and neutrality.

3. **Segmented and Detailed**: The document is divided into distinct sections and sub-sections, each addressing specific facets of AI implementation, ensuring clarity and comprehensiveness.

4. **Evidence-Based**: It uses various exhibits and data points to support claims, reinforcing an evidence-based approach.

5. **Professional and Formal**: The use of formal language and structured presentation reflects a scholarly tone aimed at business and industry professionals.

Overall, the tone maintains a balance between informative and critical, aiming to provide insights into the GenAI landscape with a focus on practical implications and future directions.

Total input tokens: 33243

Total output tokens: 1593

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [63]:

# summarization metrics
import os
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

os.environ["OPENAI_API_KEY"] = os.getenv("API_GATEWAY_KEY") or ""

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric = AnswerRelevancyMetric(
    threshold=0.7,
    include_reason=True,
    model=model,   
)

test_case = LLMTestCase(
    input=title_page,
    actual_output=author)

metric.measure(test_case)

evaluate(test_cases=[test_case], metrics=[metric])

# evalaute summary
test_case_summary = LLMTestCase(input=full_doc, actual_output=summary)
metric_summary = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the model accurately summarize the user case inputed document?",
        "Does the summary abide by the tone of African-American Vernacular English?,"
        "Does a higher score mean a more comprehensive summary?"
    ]
)

evaluate(test_cases=[test_case_summary], metrics=[metric_summary])



from IPython.display import display, Markdown
display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))



Output()

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Answer Relevancy (score: 0.8235294117647058, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.82 because while the response provides valuable insights into the GenAI Divide, it includes several irrelevant statements that stray from the core topic, such as pilot challenges and spending comparisons. These detract from the main focus, preventing a higher score, but the relevant points still contribute positively to the overall understanding., error: None)

For test case:

  - input: The GenAI Divide  
  - actual output: "The GenAI Divide" report highlights the current state of generative AI (GenAI) in business, presenting insights into the challenges and disparities in its adoption and effectiveness. Here’s a summary of the key points:

### Overview
Despite significant investments in GenAI—estimated at $30-40 billion—most organizations are not realizing substantial performance improvements, with 95% reporting no measurable impac

✓ Evaluation completed 🎉! (time taken: 18.94s | token cost: 0.00078075 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

ValueError: Number of verdicts generated does not equal.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
